# CUDA 算子优化手册

**配套练习**：`vector_add/vector_add.cu`（已编译为 `build/bin/vector_add` 并跑通）

### 这份手册是什么

一份**从头建立 CUDA 优化认知**的实战手册：不讲空理论，所有结论都用**你自己跑出来的数据**来验证。

### 写给谁

- C 语言有基础，能写简单的 CUDA kernel（比如 `vector_add`）
- 但**看不懂 profiler 输出**，也说不清「为什么慢」「该往哪优化」

### 读完你会得到什么

1. 一套**判断瓶颈**的方法：拿到任何算子，能说清它是访存受限还是算力受限
2. 一套**看懂 profiler** 的能力：知道该看哪几个数、它们意味着什么
3. 一套**可复用的操作流程（SOP）**：从 naive 实现到优化收敛
4. 一份**中英对照术语表**，以后读英文文档不再卡壳

### 阅读建议

- 第一次读：**按顺序从头读到尾**，不要跳
- 之后：把第六部分（SOP）当 checklist，把第十部分（术语表）当字典
- 遇到代码块，**动手改参数跑一遍**，比看十遍有用
- 全文术语统一写作「中文（English）」，例如 线程（thread）

### 阅读地图

整份手册的**逻辑主线**只有一条：

```mermaid
flowchart LR
    A[基础概念] --> B[硬件两条腿]
    B --> C[用 profiler 破案]
    C --> D[三种瓶颈]
    D --> E[优化方向]
    E --> F[操作流程 SOP]
```

翻译成人话：

1. **先认识机器**：GPU 和 CPU 哪里不一样、线程怎么组织、内存有哪些层级（第一部分）
2. **再认识速度上限**：机器只有「搬数据」和「算数据」两条腿（第二部分）
3. **然后破案**：把你自己的 profiler 输出一段段读懂（第三部分）
4. **归纳成模型**：瓶颈只有三种（第四部分）
5. **最后给方案**：每种瓶颈对应哪些优化手段（第五部分）
6. **固化成流程**：以后照着 SOP 做（第六部分）

## 第一部分　基础概念

### 1.1 为什么需要 GPU？——CPU 与 GPU 的根本差异

要理解 CUDA 优化，必须先接受一个反直觉的事实：

> **GPU 不是因为「单个核心更强」而快，而是因为「核心数量多得多」而快。**

两者的设计目标完全不同：

| | CPU | GPU |
| --- | --- | --- |
| 核心数量 | 几个到几十个（少） | 几千个（多） |
| 单个核心 | 又大又强，能处理复杂逻辑 | 又小又简单 |
| 擅长 | 串行逻辑、分支、低延迟 | 大规模重复计算 |
| 优化目标 | 降低**延迟** | 提高**吞吐** |

**两个必须记住的术语**

- **延迟（latency）**：一件事从开始到完成要多久。
- **吞吐（throughput）**：单位时间能完成多少件事。

**关键洞察**：GPU 靠「大量并行」来**掩盖延迟（hide latency）**。
单个线程等一次数据要等几百个周期，但只要有足够多的线程可以切换，机器就不会闲。

> 这是后面所有优化的思想源头：**GPU 的性能问题，往往不是「算得慢」，而是「在等」。**

#### 1.1.1 一个生活化的类比

- **CPU = 一辆重型卡车**：马力大、能拉复杂货物、单趟很快。
- **GPU = 一支快递车队**：每辆车很小，但有一万辆同时出发，总吞吐远超一辆卡车。

如果任务只是「搬一件复杂家具」（复杂串行逻辑），卡车更快；
如果是「送一万个小包裹」（大规模相同计算），车队碾压。

深度学习里的矩阵运算，正是「一万个小包裹」，所以要用 GPU。

> 反过来也要记住：**如果任务天然串行、分支复杂，GPU 反而会慢。** 不是所有代码都适合搬到 GPU。

### 1.2 GPU 怎么把「并行」组织起来？

CUDA 用一套**层级结构（hierarchy）**管理成千上万个线程。从大到小：

| 层级 | 英文 | 说明 |
| --- | --- | --- |
| 网格 | grid | 一次 kernel 启动产生的**全部线程** |
| 线程块 | block（thread block） | grid 被切成的若干组；**同一 block 内可以共享内存、可以同步** |
| 线程束 | warp | **32 个线程**一组，是 GPU **真正调度和执行的单位** |
| 线程 | thread | 最小执行单位 |

在 kernel 里，每个线程靠这几个**内置变量（built-in variables）**知道「我是谁」：

| 变量 | 含义 |
| --- | --- |
| `threadIdx.x` | 我在本 block 内的编号 |
| `blockIdx.x` | 我所在的 block 编号 |
| `blockDim.x` | 每个 block 有多少线程 |
| `gridDim.x` | 一共有多少个 block |

你的 `vector_add` 就是靠它们定位数据的：

```cpp
int idx = blockIdx.x * blockDim.x + threadIdx.x;   // 全局线程编号
```

启动时写成 `<<<grid, block>>>`：第一个数是**有多少个 block**，第二个数是**每个 block 有多少线程**。

#### 软件层级：grid → block → warp → thread

```mermaid
flowchart TD
    G[grid 网格 一次 kernel 启动的全部线程] --> B0[block 0 线程块]
    G --> B1[block 1]
    G --> B2[block ...]
    B0 --> W0[warp 0 线程束=32 个线程]
    B0 --> W1[warp 1]
    W0 --> T0[thread 0]
    W0 --> T1[thread 1]
    W0 --> T2[thread ... 31]
```

#### 硬件层级：GPU → SM → warp scheduler → core

```mermaid
flowchart TD
    GPU[GPU 整块显卡] --> S0[SM 0 流多处理器]
    GPU --> S1[SM 1]
    GPU --> S2[SM ... 你的卡共 24 个]
    S0 --> WS[warp scheduler warp 调度器]
    WS --> CORE[CUDA core 计算核心]
    WS --> TC[Tensor Core 张量核心]
    S0 --> L1[L1 缓存 / shared memory 共享内存]
```

两个关键术语：

- **SM（Streaming Multiprocessor，流多处理器）**：GPU 的「小 CPU 核」。你的 4060 有 **24 个**，block 会被分配到 SM 上执行。
- **warp scheduler（warp 调度器）**：每个 SM 有 4 个。它每个周期挑一个**就绪的** warp 发射指令。
  - 如果所有 warp 都在**等数据**，就没有可发射的 warp，硬件只能空转。
  - **这正是你 profiler 里发生的事情**，第三部分会逐段看。

### 1.3 内存层级（memory hierarchy）

这是**最重要的一节**，因为 GPU 优化八成的功夫都在这里。

| 层级 | 英文 | 谁能用 | 容量 | 速度 |
| --- | --- | --- | --- | --- |
| 寄存器 | register | 单个线程私有 | 极小（每线程几十个） | 最快 |
| 共享内存 | shared memory | **同一 block 内共享** | 每 SM 约 100 KB | 很快 |
| L1 / L2 缓存 | L1 / L2 cache | 硬件自动管理，程序员不直接控制 | L1 小、L2 几 MB | 较快 |
| 全局显存 | global memory（即 DRAM / HBM） | 所有线程 | 8 GB | **最慢**（几百个周期延迟） |

**三个必须记住的结论**

1. **寄存器（register）最快**：编译器会尽量把变量放进寄存器。
2. **共享内存（shared memory）是「手工缓存」**：你可以显式把数据搬进来，让同一 block 的线程**复用**它。这是后面「分块（tiling）」的基础。
3. **全局显存（global memory）最慢**：每访问一次要等几百个周期。**所有 memory-bound 问题的根源都在这里。**

> 一句话：**优化的本质，是让数据尽量在「快」的层级被复用，少去碰「慢」的全局显存。**

### 1.4 一个 kernel 的完整生命周期

```mermaid
flowchart LR
    A[CPU 发起 kernel 启动] --> B[grid 进入 GPU 调度队列]
    B --> C[block 被分配到空闲的 SM]
    C --> D[SM 把 block 拆成若干 warp]
    D --> E[warp scheduler 轮流发射指令]
    E --> F[线程以 SIMT 方式执行]
    F --> G[访存请求发往 L1 再到 L2 再到 DRAM]
    G --> H[结果写回全局显存]
```

再补两个术语：

- **SIMT（Single Instruction Multiple Threads，单指令多线程）**：一个 warp 内的 32 个线程，**同一条指令**一起执行，但各自操作自己的数据。
- **合并访存（coalescing）**：如果一个 warp 的 32 个线程访问的是**连续地址**，硬件能把多次请求合并成少数几次大事务，从而大幅提速。
  - 你的 `vector_add` 里 `a[i]`、`b[i]`、`c[i]` 都是连续访问，所以访存效率很高——这是它能跑到峰值带宽 96% 的原因之一。

> **第一部分小结**：GPU 靠海量线程掩盖延迟 → 线程被组织成 warp / block / grid → 数据存放在不同层级的存储里 → **全局显存最慢，是性能陷阱的主要来源**。

## 第二部分　硬件只有两条「腿」

### 2.1 搬数据 vs 算数据

任何程序在硬件上只做两件事：

1. **搬数据**：从显存搬到芯片，算完再搬回去。上限是**显存带宽（memory bandwidth）**，单位 GB/s。
2. **算数据**：加减乘除。上限是**算力（compute throughput）**，单位 FLOP/s。

先解释几个单位：

- **FLOP（Floating-point Operation，浮点运算）**：一次浮点加、减、乘、除。
- **FLOP/s（Floating-point Operations Per Second）**：每秒能做多少次浮点运算。
- **GB/s（Gigabytes per second）**：每秒能搬多少字节。

> **优化的全部工作，就是找出这两条腿里哪条短，然后把它加长。**

### 2.2 你这张卡的两条腿有多长

不用查规格书，直接从你自己的 profiler 输出里反推：

| profiler 里的字段 | 数值 | 含义 |
| --- | --- | --- |
| `DRAM Frequency` | 8.01 cycle/ns | 显存时钟跑在 8.01 GHz |
| `SM Frequency` | 1.55 cycle/ns | 计算核心跑在 1.55 GHz |
| `Max Bandwidth` | 90.76 % | 已经达到带宽上限的 90.76% |

**显存带宽怎么算**

GDDR6 是**双倍数据率（DDR, Double Data Rate）**，总线宽度 128 bit = 16 字节：

    峰值带宽 = 8.01 GHz × 2（因为 DDR）× 16 字节 ≈ 256.3 GB/s

**算力怎么算**

你的卡：24 个 SM，每个 SM 有 128 个 FP32 核心，每个核心每周期能做 2 次浮点运算（FMA）：

    峰值算力 = 24 × 128 × 2 × 1.55 GHz ≈ 9.5 TFLOP/s

顺便记住两个词：

- **FMA（Fused Multiply-Add，融合乘加）**：一次完成 `a * b + c`，算 2 次运算但只花 1 条指令。
- **TFLOP/s**：每秒一万亿次浮点运算。

### 2.3 两个比值，决定一切

**算子侧：算术强度（Arithmetic Intensity，AI）**

    AI = 运算次数(FLOP) ÷ 搬运字节数(Bytes)      单位：FLOP/Byte

它衡量「这个算子**每搬 1 字节数据，能算多少次运算**」。

以你的 `vector_add` 为例，处理 1 个元素：

| 动作 | 流量 |
| --- | --- |
| 读 `a[i]` | 4 字节 |
| 读 `b[i]` | 4 字节 |
| 写 `c[i]` | 4 字节 |
| 运算 `a[i] + b[i]` | 1 FLOP |

    AI = 1 ÷ 12 ≈ 0.083 FLOP/Byte

> 注意：AI 是**算子自身的属性，和硬件无关**。换一张显卡，这个 0.083 不变。

**硬件侧：机器平衡点（Machine Balance）**

    机器平衡点 = 峰值算力 ÷ 峰值带宽 = 9.5 TFLOP/s ÷ 256 GB/s ≈ 37 FLOP/Byte

它衡量「这台机器**每搬 1 字节，顺便能算多少次运算**」。

**两者一比，瓶颈就出来了**

| 比较 | 英文术语 | 中文含义 |
| --- | --- | --- |
| AI < 平衡点 | **memory-bound** | 访存受限：在等数据 |
| AI > 平衡点 | **compute-bound** | 算力受限：在等计算 |

`vector_add` 的 AI = 0.083，远远小于 37 → **memory-bound**。

直觉：**每搬 1 字节，你能顺便算出 37 次运算吗？**
- 不能 → 你在等数据（memory-bound）
- 能 → 你在等算力（compute-bound）

#### 用代码把这些数字算出来

下面定义几个工具函数，**后面所有代码块都会复用**。

> 建议：跑完之后**自己改改数字**（比如把带宽改成 A100 的 1555 GB/s），看看结论会不会变。

In [ ]:
# ===== 机器参数（从 profiler 实测值反推）=====
PEAK_BW    = 256.3e9     # 显存带宽上限，字节/秒
PEAK_FLOPS = 9.5e12      # FP32 算力上限，FLOP/秒

BALANCE = PEAK_FLOPS / PEAK_BW   # 机器平衡点，单位 FLOP/Byte


def intensity(flops, bytes_moved):
    # 算术强度：每搬 1 字节，能算多少次运算
    return flops / bytes_moved


def bound_of(ai):
    # 根据算术强度判断瓶颈类型
    if ai < BALANCE * 0.5:
        return 'memory-bound  访存受限'
    if ai > BALANCE * 2.0:
        return 'compute-bound 算力受限'
    return 'balanced      接近平衡点'


def lower_bound_ms(flops, bytes_moved):
    # 理论耗时下界 = max(搬数据时间, 算数时间)
    t_mem = bytes_moved / PEAK_BW
    t_cmp = flops / PEAK_FLOPS
    return t_mem * 1e3, t_cmp * 1e3, max(t_mem, t_cmp) * 1e3


print(f'机器平衡点 = {BALANCE:.1f} FLOP/Byte')
print(f'含义：每搬 1 字节数据，机器顺便能算 {BALANCE:.0f} 次运算')

In [ ]:
# ===== 四个算子的算术强度对比 =====
# 每个条目：(总 FLOP, 总搬运字节数)
ops = {
    'vector_add  读2写1':        (1, 12),
    'dot_product 向量点积':       (2, 8),
    'softmax     单元素粗估':     (5, 8),
    'GEMM 4096   naive':         (2 * 4096**3, 3 * 4096**2 * 4),
}

for name, (f, b) in ops.items():
    ai = intensity(f, b)
    tm, tc, lb = lower_bound_ms(f, b)
    print(f'{name:<24} AI = {ai:>10.4f} FLOP/Byte   {bound_of(ai):<26} 时间下界 {lb:>8.3f} ms')

print()
print('规律：只要「每个元素只算几次，却要搬好几个字节」，就一定是 memory-bound。')
print('AI Infra 里绝大多数算子都是这一类。')

## 第三部分　破案：逐段读懂你自己的 profiler 输出

下面把你在 `vector_add.cu` 底部记录的那次 `ncu` 输出**原样贴进来**，一段一段读。
每一段都是「**原始输出** → **逐项解读** → **结论**」的结构。

命令回顾：

```bash
ncu --set full -k vectorADD --launch-count 1 ./build/bin/vector_add
```

这次运行的头部信息：

```text
[132819] vector_add@127.0.0.1
  vectorADD(const float *, const float *, float *, int) (262144, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 8.9
```

**怎么读这行**：kernel 名是 `vectorADD`；`(262144, 1, 1)` 是 **grid**（262144 个 block）；`x(256, 1, 1)` 是 **block**（每个 block 256 个线程）；`CC 8.9` 是计算能力 8.9。

**术语**：
- **profiler**：性能分析器，这里用的是 NVIDIA Nsight Compute（命令 `ncu`）。
- **kernel**：在 GPU 上并行执行的函数，用 `__global__` 声明。
- **`--launch-count 1`**：只分析第 1 次 kernel 启动，避免重复采样。

### 3.1 GPU Speed Of Light：两个方向各用了多少

**原始输出**

```text
    Section: GPU Speed Of Light Throughput
    ----------------------- ------------- ------------
    Metric Name               Metric Unit Metric Value
    ----------------------- ------------- ------------
    DRAM Frequency          cycle/nsecond         8.01
    SM Frequency            cycle/nsecond         1.55
    Elapsed Cycles                  cycle    5,268,927
    Memory Throughput                   %        90.76
    DRAM Throughput                     %        90.76
    Duration                      msecond         3.40
    L1/TEX Cache Throughput             %        17.11
    L2 Cache Throughput                 %        21.30
    SM Active Cycles                cycle 5,268,893.88
    Compute (SM) Throughput             %        13.27
    ----------------------- ------------- ------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `DRAM Frequency` | 8.01 cycle/ns | 显存时钟 8.01 GHz；×2（DDR）×16 字节 ≈ 256 GB/s |
| `SM Frequency` | 1.55 cycle/ns | 计算核心频率，用来反推算力上限 |
| `Duration` | 3.40 ms | 本次 kernel 的耗时 |
| `Memory Throughput` | **90.76 %** | 显存带宽使用率 |
| `DRAM Throughput` | **90.76 %** | 同上，**这一行是判定的关键** |
| `L1/TEX Cache Throughput` | 17.11 % | 片上一级缓存通路占用 |
| `L2 Cache Throughput` | 21.30 % | 二级缓存通路占用 |
| `Compute (SM) Throughput` | **13.27 %** | 算力使用率 |

**结论**

> 访存 **91%**，算力 **13%** → 瓶颈明确在显存侧，属于 **memory-bound（访存受限）**。
> 这和第二部分用算术强度算出的结论完全一致（AI 0.083 « 平衡点 37），两条独立路径互相印证。

### 3.2 Compute Workload Analysis：算力为什么闲着

**原始输出**

```text
    Section: Compute Workload Analysis
    -------------------- ----------- ------------
    Metric Name          Metric Unit Metric Value
    -------------------- ----------- ------------
    Executed Ipc Active   inst/cycle         0.33
    Executed Ipc Elapsed  inst/cycle         0.33
    Issue Slots Busy               %         8.29
    Issued Ipc Active     inst/cycle         0.33
    SM Busy                        %         8.29
    -------------------- ----------- ------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Executed Ipc Active` | 0.33 inst/cycle | 每周期只执行 0.33 条指令（理想接近 1）|
| `Executed Ipc Elapsed` | 0.33 inst/cycle | 同上，按整个 kernel 时长平均 |
| `Issue Slots Busy` | 8.29 % | 发射槽只有 8% 的时间在用 |
| `SM Busy` | 8.29 % | SM 九成时间在等 |

**术语**

- **IPC（Instructions Per Cycle，每周期指令数）**：衡量流水线利用率，越接近 1 越忙。

**结论**

> 算力侧严重闲置。**但这不是问题**——瓶颈在访存，算力本来就不该忙。
> 后面第五部分会讲：把非瓶颈资源推到 100% 毫无收益。

### 3.3 Memory Workload Analysis：访存侧的细节

**原始输出**

```text
    Section: Memory Workload Analysis
    --------------------------- ------------ ------------
    Metric Name                  Metric Unit Metric Value
    --------------------------- ------------ ------------
    Memory Throughput           Gbyte/second       232.61
    Mem Busy                               %        15.98
    Max Bandwidth                          %        90.76
    L1/TEX Hit Rate                        %            0
    L2 Compression Success Rate            %            0
    L2 Compression Ratio                                0
    L2 Hit Rate                            %        33.38
    Mem Pipes Busy                         %        13.27
    --------------------------- ------------ ------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Memory Throughput` | 232.61 GB/s | 实测带宽；232.61 ÷ 256.3 ≈ 90.76%，与上一段吻合 |
| `Max Bandwidth` | 90.76 % | 距离带宽上限只剩 9% |
| `L1/TEX Hit Rate` | **0 %** | 每个元素只读一次，**完全没有复用**，符合预期 |
| `L2 Hit Rate` | 33.38 % | 主要来自写 `c` 时的缓存行为，**不是真正的数据复用** |
| `Mem Pipes Busy` | 13.27 % | 访存流水线本身不忙，忙的是 DRAM 带宽 |
| `L2 Compression` | 0 | 压缩未启用 |

**术语**

- **缓存命中率（cache hit rate）**：访存请求在缓存里找到数据的比例，越高越省显存带宽。
- **复用（reuse）**：同一份数据被多次使用。`vector_add` 每个元素只用一次，没有复用。

**结论**

> L1 命中率 0% 说明数据是**「流过」**的：读完就扔，完全靠带宽硬扛。
> 这不是错误——`vector_add` 本该是「流式（streaming）」的。
> 但也说明它**没靠缓存省下任何流量**，所以唯一能做的就是把搬运本身做得更快（向量化）。

### 3.4 Scheduler Statistics：调度器在等什么

**原始输出**

```text
    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %         8.28
    Issued Warp Per Scheduler                        0.08
    No Eligible                            %        91.72
    Active Warps Per Scheduler          warp         9.69
    Eligible Warps Per Scheduler        warp         0.09
    ---------------------------- ----------- ------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Active Warps Per Scheduler` | 9.69 warp | 每个调度器平均挂着约 10 个 warp |
| `Eligible Warps Per Scheduler` | **0.09 warp** | 但其中只有 0.09 个是「就绪」的 |
| `Issued Warp Per Scheduler` | **0.08** | 每周期只发射 0.08 条（理想 1）|
| `No Eligible` | **91.72 %** | **九成以上的周期，没有任何 warp 可发射** |
| `One or More Eligible` | 8.28 % | 只有 8% 的周期有活可干 |

**术语**

- **eligible（就绪）**：warp 的下一条指令所需条件都满足了，随时可以发射。
- **issue（发射）**：调度器把一条指令送进执行流水线。

**结论**

> **warp 不缺（挂着 9.69 个），缺的是「就绪」的 warp**。
> 说明它们都在等同一个东西——数据。这就是下面 3.5 要看的。

### 3.5 Warp State Statistics：到底在等什么

**原始输出**

```text
    Section: Warp State Statistics
    ---------------------------------------- ----------- ------------
    Metric Name                              Metric Unit Metric Value
    ---------------------------------------- ----------- ------------
    Warp Cycles Per Issued Instruction             cycle       117.00
    Warp Cycles Per Executed Instruction           cycle       117.02
    Avg. Active Threads Per Warp                                   32
    Avg. Not Predicated Off Threads Per Warp                    28.80
    ---------------------------------------- ----------- ------------

    WRN   On average, each warp of this kernel spends 108.3 cycles being stalled waiting for a scoreboard dependency on
          a L1TEX (local, global, surface, texture, rtcore) operation. This represents about 92.6% of the total
          average of 117.0 cycles between issuing two instructions.
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Warp Cycles Per Issued Instruction` | 117.00 cycle | 平均每条指令要等 117 个周期 |
| 其中等待 `L1TEX` 的 | **108.3 cycle** | 占 92.6%，**几乎全在等显存数据回来** |
| `Avg. Active Threads Per Warp` | 32 | warp 满员，没有线程退出 |
| `Avg. Not Predicated Off Threads Per Warp` | 28.80 | 平均 28.8 个线程真正在干活（其余被谓词屏蔽）|

**术语**

- **stall（停滞）**：warp 想执行下一条指令，但条件不满足（比如数据没回来），只能等。
- **scoreboard dependency（记分牌依赖）**：硬件记录「这条指令的结果还没到，后面的指令必须等」。
- **L1TEX**：L1 缓存 / 纹理单元这条访存通路。等 L1TEX 就是**在等访存结果**。
- **predicated off（谓词屏蔽）**：指令被条件关掉不执行，对应线程空转。

**结论**

> 现象：**挂着近 10 个 warp，但几乎全在等数据**，调度器 92% 的周期无事可做。
> 这叫**延迟受限（latency-bound）**——不是带宽不够，而是**并发还不够多**，没能把访存延迟完全藏住。
> 所以 `vector_add` 是 **memory-bound + latency-bound 的混合体**：带宽用到 91%，剩下 9% 的缺口来自延迟。

### 3.6 Occupancy：占用率不是瓶颈

**原始输出**

```text
    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           24
    Block Limit Registers                 block           16
    Block Limit Shared Mem                block           16
    Block Limit Warps                     block            6
    Theoretical Active Warps per SM        warp           48
    Theoretical Occupancy                     %          100
    Achieved Occupancy                        %        81.04
    Achieved Active Warps Per SM           warp         38.90
    ------------------------------- ----------- ------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Theoretical Occupancy` | 100 % | 理论上可以坐满 48 个 warp |
| `Achieved Occupancy` | 81.04 % | 实际驻留 38.9 个 warp |
| `Block Limit Warps` | 6 block | block=256（8 warp）时，一个 SM 最多放 6 个 block |
| `Block Limit Registers` / `Shared Mem` | 16 block | 寄存器与共享内存都没有成为限制 |

**术语**

- **占用率（occupancy）**：SM 上**实际驻留的 warp 数 ÷ 最大可驻留 warp 数**。通俗说就是「工位坐满了多少」。

**结论**

> 占用率 81%，不低；而且**提高它对 memory-bound kernel 帮助有限**。
> 最有力的反证：你们后面扫 block size 时，**block=1024 的理论占用率反而更低，速度却更快**。
> 这直接证明：**占用率不是这个 kernel 的瓶颈。**

### 3.7 Launch Statistics：启动参数告诉我们什么

**原始输出**

```text
    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                262,144
    Registers Per Thread             register/thread              16
    Shared Memory Configuration Size           Kbyte           16.38
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    Threads                                   thread      67,108,864
    Waves Per SM                                            1,820.44
    -------------------------------- --------------- ---------------
```

**逐项解读**

| 指标 | 你的数值 | 说明 |
| --- | --- | --- |
| `Block Size` / `Grid Size` | 256 / 262144 | 与启动时 `<<<grid, block>>>` 一致：262144 × 256 = 67,108,864 |
| `Threads` | 67,108,864 | 正好等于 N = 2²⁶，说明**每个线程恰好处理 1 个元素** |
| `Registers Per Thread` | 16 register | 很省，不会限制占用率 |
| `Static/Dynamic Shared Memory Per Block` | 0 byte | 这个 kernel **完全没用共享内存** |
| `Waves Per SM` | 1,820.44 | block 总量相当于 SM 容量的 1820 倍 |

**术语**

- **waves per SM（波次）**：一个 kernel 的 block 数相当于 SM 容量的多少倍。数值很大说明要「翻台」很多轮。
- **寄存器（register）**：线程私有的最快存储；数量越多，能同时驻留的 warp 越少。

**结论**

> **wave = 1820** 说明任务量远大于硬件容量，最后一批 block 收尾的开销可以忽略，机器全程都很忙。
> 同时 `Shared Memory = 0` 也确认了：这个 kernel 没有任何片上复用，纯粹的流式搬运。

### 3.8 四条提示与警告：哪些该听，哪些要无视

**原始输出**

```text
    INF   The kernel is utilizing greater than 80.0% of the available compute or memory performance of the device. To
          further improve performance, work will likely need to be shifted from the most utilized to another unit.
          Start by analyzing DRAM in the Memory Workload Analysis section.

    WRN   All compute pipelines are under-utilized. Either this kernel is very small or it doesn't issue enough warps
          per scheduler. Check the Launch Statistics and Scheduler Statistics sections for further details.

    WRN   Every scheduler is capable of issuing one instruction per cycle, but for this kernel each scheduler only
          issues an instruction every 12.1 cycles. This might leave hardware resources underutilized and may lead to
          less optimal performance. Out of the maximum of 12 warps per scheduler, this kernel allocates an average of
          9.69 active warps per scheduler, but only an average of 0.09 warps were eligible per cycle.

    WRN   This kernel executes 0 fused and 2097152 non-fused FP32 instructions. By converting pairs of non-fused
          instructions to their fused, higher-throughput equivalent, the achieved FP32 performance could be
          increased by up to 50%
```

**逐条甄别**

| 提示 / 警告 | 要不要听 | 理由 |
| --- | --- | --- |
| `INF` 使用了超过 80% 的带宽或算力 | ✅ **要听** | 说明已接近上限，该换方向，而不是继续抠这个 kernel |
| `WRN` All compute pipelines are under-utilized | ❌ **无视** | memory-bound kernel 本来就算力闲置，这是**结果**不是**原因** |
| `WRN` 每个调度器 12.1 周期才发一条 | ⚠️ **参考** | 描述的是「等待」的后果，根因已在 3.5 定位（等 L1TEX）|
| `WRN` 用 FMA 可提升 50% | ❌ **误导** | 这个 kernel 只有 `a + b`，**根本没有乘法，没有 FMA 可融合** |

**术语**

- **INF / WRN**：profiler 的提示级别（Information / Warning），只是建议，不是错误。

> **重要认知**：profiler 的建议是**通用模板**，必须结合 kernel 的语义判断。
> 看到「可提升 50%」这种话不要兴奋，先问一句：这个建议对我的代码成立吗？

### 3.9 破案结论

把第三部分串起来：

| 观察 | 数据 | 结论 |
| --- | --- | --- |
| 带宽 vs 算力 | 90.76% vs 13.27% | 瓶颈是**访存** → memory-bound |
| warp 在等什么 | 92.6% 的周期在等 L1TEX | 等显存 → **延迟受限** |
| 缓存命中 | L1 = 0% | 无复用，纯流式 |
| 占用率 | 81% | 不构成瓶颈 |
| 波次 | 1820 | 任务量充足，无收尾浪费 |

**一句话**：`vector_add` 是「带宽几乎跑满、但被访存延迟拖住最后 9%」的典型流式 kernel。

下面用代码把「实测 vs 理论」摊开对比。

In [ ]:
# ===== 实测 vs 理论：vector_add =====
N = 2**26
bytes_moved = 3 * N * 4          # 读 a + 读 b + 写 c
flops = N                        # 每个元素 1 次加法

tm, tc, lb = lower_bound_ms(flops, bytes_moved)

measured_ms  = 3.276             # block=1024 的最好成绩
measured_bw  = 245.8e9
ncu_bw       = 232.61e9          # profiler 报的带宽
ncu_dram_pct = 90.76

print(f'理论下界: 搬数据 {tm:.3f} ms | 算数 {tc:.5f} ms | 取大者 {lb:.3f} ms')
print(f'实测最优: {measured_ms:.3f} ms  ->  达到理论下界的 {lb / measured_ms * 100:.1f}%')
print(f'实测带宽: {measured_bw / 1e9:.1f} GB/s = 峰值的 {measured_bw / PEAK_BW * 100:.1f}%')
print(f'profiler: {ncu_bw / 1e9:.2f} GB/s, DRAM Throughput {ncu_dram_pct}%')
print()
print(f'搬数据要 {tm:.2f} ms，算数只要 {tc:.5f} ms，相差约 {tm / tc:.0f} 倍')
print('=> 耗时由「搬数据」决定，所以是 memory-bound')

## 第四部分　瓶颈只有三种

把前面的分析归纳成一张表。**以后拿到任何 kernel，先做这个分类。**

| 瓶颈类型 | 英文术语 | profiler 特征 | 典型算子 | 优化大方向 |
| --- | --- | --- | --- | --- |
| 访存受限 | **memory-bound** | DRAM 高、SM 低 | vector_add、softmax、ReLU | 减少访存 / 提高访存效率 / 算子融合 |
| 算力受限 | **compute-bound** | SM 高、DRAM 低 | GEMM、卷积 | 提高数据复用 / 用 Tensor Core |
| 延迟受限 | **latency-bound** | **两个都低** | 未优化的 reduction、超小 kernel | 提高并发 / 减少同步 |

### 为什么有第三类？

如果 DRAM 和 SM **都很低**（比如 30% 和 10%），说明瓶颈既不是带宽也不是算力，而是：

- **延迟（latency）**：线程不够多，掩盖不住访存延迟
- **同步 / 依赖**：`__syncthreads` 太频繁，或线程之间互相等

**你已经在 profiler 里见过第三类的影子**：3.5 里「92.6% 的周期在等 L1TEX」就是延迟效应。
`vector_add` 靠**大量 warp**把延迟盖住了大部分，所以带宽还能到 91%；
一旦 warp 不够（block 太小、寄存器太多、kernel 太短），就会真正掉进 latency-bound，带宽可能只剩 30%。

> 所以优化顺序永远是：**先分类，再动手。** 分错了类，努力全白费。

In [ ]:
# ===== 一个小「诊断器」：给 profiler 的两个百分比，判断瓶颈 =====
def diagnose(dram_pct, sm_pct):
    if dram_pct > 60 and sm_pct < 50:
        return 'memory-bound    访存受限：优化访存效率'
    if sm_pct > 60 and dram_pct < 50:
        return 'compute-bound   算力受限：提高数据复用'
    if dram_pct < 60 and sm_pct < 60:
        return 'latency-bound   延迟受限：提高并发/减少同步'
    return 'balanced        两者都接近上限'


cases = [
    ('vector_add 你的实测', 90.76, 13.27),
    ('优化良好的 GEMM', 12.00, 88.00),
    ('未优化的 reduction', 30.00, 10.00),
]

for name, dram, sm in cases:
    print(f'{name:<22} DRAM={dram:>6}%  SM={sm:>6}%  ->  {diagnose(dram, sm)}')

## 第五部分　优化方向

### 5.1 第一性原理

所有优化都遵循同一句话：

> **找出瓶颈资源，把它推到接近 100%。**

推**非瓶颈**资源毫无意义：`vector_add` 就算把算力从 13% 提到 100%，时间也不会变。

下面按三种瓶颈分别给手段。

### 5.2 memory-bound 的三大手段

**手段一：减少访存总量**

- **算子融合（operator fusion）**：把多个算子合成一个 kernel，避免中间结果写回显存再读出来。
  - 例：`ReLU(x + b)` 如果分两步，要读写显存 4 次；融合后只需 2 次。
  - 这是**收益最大**的方向，也是 FlashAttention 的核心思想。

**手段二：提高访存效率**

- **合并访存（coalescing）**：保证一个 warp 内的 32 个线程访问连续地址。
- **向量化访存（vectorization）**：用 `float4` 让一个线程一次搬 16 字节，而不是 4 字节。
  - 效果：访存指令数减少到 1/4，更容易吃满带宽。
  - **这是你的 `vector_add` 下一步最值得试的优化。**

**手段三：提高数据复用**

- **分块（tiling）**：把数据先搬到**共享内存（shared memory）**，让同一 block 的线程反复使用，减少对全局显存的访问。
  - 注意：`vector_add` **无复用可挖**，所以这一招对它无效——用不上就不要硬用。

### 5.3 compute-bound 的三大手段

**手段一：提高算术强度（提高数据复用）**

- **寄存器分块（register blocking）**：一个线程负责输出**一小块**结果，中间结果留在寄存器里反复使用。
- **共享内存分块（shared memory tiling）**：经典的 GEMM 优化，把 A、B 的分块放到共享内存里复用。

**手段二：用 Tensor Core 和低精度**

- **Tensor Core（张量核心）**：专做矩阵乘加的特殊硬件，吞吐远高于普通 CUDA core。
- **低精度（reduced precision）**：`FP16`、`BF16`、`TF32` 比 `FP32` 快得多，显存占用也减半。
  - 代价：精度下降，需要用混合精度策略（关键部分仍用 FP32 累加）。

**手段三：减少冗余计算**

- 用算法换计算量、合并常量、提前化简表达式。

> 注意：**compute-bound 的优化方向，和 memory-bound 几乎完全相反。**
> GEMM 要的是「把数据留在片上反复用」，`vector_add` 要的是「把数据快速冲过去」。

### 5.4 latency-bound 的手段

| 手段 | 说明 |
| --- | --- |
| 提高占用率（occupancy） | 减小每线程寄存器数、减小 block 大小，让更多 warp 驻留 |
| 提高指令级并行（ILP, Instruction-Level Parallelism） | 一个线程一次处理多个元素，让访存请求重叠 |
| 减少同步（synchronization） | 少用 `__syncthreads`，避免 `__shared__` 乱用 |
| 增大任务粒度 | 太小的 kernel 填不满机器，考虑合并或每线程做多个元素 |

> 判断依据：**DRAM 和 SM 都低** → 大概率是这一类。

### 5.5 Roofline 模型：一眼看上限

把「算术强度」当横轴、「性能」当纵轴，就得到 **Roofline（屋顶线）** 模型：

```text
性能(FLOP/s)
  ^
  |                   ______________  <- 算力天花板（横向）
  |                  /
  |                 /   compute-bound 区
  |                /
  |   memory-bound/
  |    ______/  <- 带宽天花板（斜线，斜率 = 带宽）
  |   /
  |  /
  +-----------------------------> 算术强度 (FLOP/Byte)
                 ^
            机器平衡点
```

两条天花板：
- **斜线**：受带宽限制时，性能 = 带宽 × 算术强度
- **水平线**：受算力限制时，性能 = 峰值算力

**用途**：把算子的 AI 放到横轴上，就能立刻看出它的理论上限。
如果实测性能**远低于**天花板，说明问题不在硬件上限，而在延迟 / 占用率 / 访存模式。

In [ ]:
# ===== 画出你这张卡的 Roofline（需要 matplotlib，缺失则跳过）=====
try:
    import matplotlib.pyplot as plt

    ai_x = [10 ** e for e in [i / 10 for i in range(-10, 41)]]
    roof = [min(PEAK_BW * a, PEAK_FLOPS) for a in ai_x]

    plt.figure(figsize=(8, 5))
    plt.loglog(ai_x, roof, linewidth=2, label='Roofline ceiling')
    plt.axvline(BALANCE, linestyle='--', color='gray')
    plt.text(BALANCE * 1.4, PEAK_FLOPS * 3, 'Machine balance ~37', color='gray')

    plt.scatter([0.083], [0.083 * PEAK_BW], color='red', zorder=5)
    plt.text(0.1, 0.083 * PEAK_BW * 0.5, 'vector_add (AI=0.083)', color='red')

    plt.scatter([683], [PEAK_FLOPS], color='green', zorder=5)
    plt.text(120, PEAK_FLOPS * 0.25, 'GEMM 4096 (AI=683)', color='green')

    plt.xlabel('Arithmetic Intensity (FLOP/Byte)')
    plt.ylabel('Performance (FLOP/s)')
    plt.title('RTX 4060 Laptop Roofline')
    plt.grid(True, which='both', alpha=0.3)
    plt.legend()
    plt.show()
except ImportError:
    print('未安装 matplotlib，跳过绘图。安装命令: pip install matplotlib')

### 5.6 回到 vector_add：还能优化吗？

| 手段 | 对 `vector_add` 有没有用 | 原因 |
| --- | --- | --- |
| 算子融合 | ❌ 已是最小粒度 | 它本身就是最基础的算子 |
| 共享内存分块 | ❌ 无复用 | 每个元素只读一次 |
| **向量化（float4）** | ✅ **值得试** | 减少访存指令数，可能再榨 1～2% |
| 调 block size | ✅ 已试过 | block=1024 拿到 96%，接近平台期 |

**结论**

> `vector_add` 已经跑到**峰值带宽的 96%**，剩下 4% 是 DRAM 的固有开销（刷新、读写切换）。
> **这是一个重要的能力：知道什么时候优化已经到头了。**
> 继续抠它性价比极低；正确做法是往上走——去优化**真正有优化空间**的算子（GEMM、reduction、softmax）。

## 第六部分　操作流程：从零到一个优化好的算子（SOP）

这是**可复用的标准流程**，每一步都有明确的「完成标准」。

| # | 步骤 | 完成标准 |
| --- | --- | --- |
| 1 | 明确算子语义与精度要求 | 能用一句话说清输入、输出 |
| 2 | 估算算术强度 AI | 得到一个 FLOP/Byte 的粗略数字 |
| 3 | 判断瓶颈（对机器平衡点） | 知道该往哪个方向优化 |
| 4 | 实现 naive baseline | 能跑、结果正确 |
| 5 | 正确性对拍 | 与 CPU 参考实现一致（含边界） |
| 6 | 可靠计时 | warm-up + cudaEvent + 多次取最小值 |
| 7 | profiler 分析 | 拿到 DRAM% / SM% / stall 原因 |
| 8 | 定位瓶颈并优化 | **每次只改一个变量** |
| 9 | 回到第 5 步，记录数据 | 有对比表格 / 曲线 |

**第 4～9 步是一个循环**：改一次、测一次、记录一次。**一次只改一个变量**，否则你分不清是哪个改动起了作用。

### 细节 ①：每个工程都要有错误检查宏

不检查返回值，出错时你只会看到「结果不对」，而不是「哪一行错了」。

```cpp
#define CUDA_CHECK(call)                                          \
    do {                                                          \
        cudaError_t err = (call);                                 \
        if (err != cudaSuccess) {                                 \
            fprintf(stderr, "CUDA error %s:%d: %s\n",             \
                    __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                   \
        }                                                         \
    } while (0)
```

> **踩坑提醒**：宏的每一行末尾必须有行连接符 `\`（一个反斜杠），否则后面的行会脱离宏体，
> 报出一堆莫名其妙的 `expected a declaration`。这个坑我们真的踩过。

### 细节 ②：计时必须用 cudaEvent，不能用 CPU 计时

kernel 是**异步（asynchronous）**的：CPU 发出启动命令后就返回了，此时 kernel 可能还没跑完。
用 CPU 计时会严重低估耗时。`cudaEvent` 记在 **GPU 自己的时间线**上，才准确。

### 细节 ③：必须先 warm-up

第一次运行会触发 GPU 提频、加载指令缓存，耗时偏高。先跑几次再测。

### 细节 ④：多次测量取最小值

笔记本显卡会**热降频（thermal throttling）**。实测教训：同一份代码、同一个参数，两次运行差了 **13%**（4.04 ms → 3.53 ms），纯粹是温度造成的。**单次测量不足信。**

### 细节 ⑤：浮点比较要用容差

```cpp
if (fabsf(h_c[i] - ref[i]) > 1e-5f) ++bad;   // 不要写 ==
```

### 细节 ⑥：参数扫描要交错，不要顺序

顺序扫描会被热漂移污染（跑越久越热、越热越慢），你可能误判「某个参数更差」。
改进：交错扫描（如 128 → 1024 → 256 → 512），或整轮倒序再跑一次。

## 第七部分　常见陷阱清单

| 陷阱 | 后果 | 对策 |
| --- | --- | --- |
| 单次测量 | 把热漂移当成优化效果 | 多次取最小值 |
| 同时改多个变量 | 不知道哪个改动有用 | 一次只改一个 |
| 只看平均值 | 掩盖偶发的慢 | 看最小值 / 分布 |
| 照搬 profiler 建议 | 优化了非瓶颈 | 结合 kernel 语义判断 |
| 忽略正确性 | 快但错 | 每次都跑对拍 |
| 盲目追占用率 | 对 memory-bound 无收益 | 先看 DRAM% |
| 硬套优化手段 | 用不上还添乱 | 先分类瓶颈，再选手段 |
| 抄代码不看懂 | 换个算子就不会写 | 能独立默写才算会 |

> **优化的第一性原理**：先问「瓶颈是什么」，再问「怎么把瓶颈推到 100%」。
> 跳过第一步的优化，都是碰运气。

## 第八部分　实战复盘：vector_add

### 实测数据（block 扫描，N = 2²⁶，每档 5 次取最小值）

| block | 最小时间 (ms) | 带宽 (GB/s) | 峰值占比 |
| --- | --- | --- | --- |
| 128 | 3.527 | 228.3 | 89% |
| 256 | 3.528 | 228.3 | 89% |
| 512 | 3.523 | 228.6 | 89% |
| 1024 | **3.275** | **245.9** | **96%** |

### 从数据里读出三件事

1. **128 / 256 / 512 几乎完全一样** → 这是 memory-bound 的标志：block size 影响很小。
2. **1024 明显更快（96%）** → 少量收益，但已接近天花板。
3. **扫描过程中出现的 4.03 ms 是噪声** → 热降频，不是真实性能差异。

### 完整结论

| 问题 | 答案 | 依据 |
| --- | --- | --- |
| 什么瓶颈？ | memory-bound | DRAM 91% vs SM 13%；AI 0.083 « 37 |
| 还有空间吗？ | 几乎没有 | 已到峰值带宽 96% |
| 下一步做什么？ | 换有价值的算子 | 抠 `vector_add` 性价比极低 |

> **记住两个层次的差别**：
> 对**单个 kernel**，「让 DRAM 满载」是目标；
> 对**整个模型**，「减少 DRAM 访问总量」才是目标。
> 前者是 `vector_add` 的思路，后者是算子融合 / FlashAttention 的思路。

## 第九部分　自测题

### A. 判断瓶颈（必须说出理由）

1. **softmax**：读 N 个数，做几次加减 / 除法，输出 N 个数
2. **向量点积（dot product）**：读 2N 个 float，只输出 1 个数
3. **FP16 GEMM**：用 Tensor Core 后算力约 60 TFLOP/s，机器平衡点变成 60 ÷ 0.256 ≈ 234 FLOP/Byte。此时 AI = 683 的 GEMM 还是 compute-bound 吗？

### B. 概念解释

4. 为什么说「让 DRAM 跑满」不是目标，而只是 memory-bound 情况下的**结果**？
5. 如果 profiler 显示 DRAM 30%、SM 10%，说明瓶颈是什么？该往哪查？
6. 为什么 block size 对 `vector_add` 影响很小？
7. 为什么 profiler 说「改用 FMA 可提升 50%」时要先怀疑它？
8. `Waves Per SM = 1820` 说明了什么？

### C. 动手

9. 把 `vector_add` 改成 `float4` 版本，对比带宽变化。
10. 用交错扫描重跑 block 表，验证 block=1024 的优势是否稳定。
11. 给一个你熟悉的算子（比如 ReLU）估算 AI，预测瓶颈类型，再用 profiler 验证。

## 第十部分　术语表（中英对照）

### 硬件与执行

| 术语 | 含义 |
| --- | --- |
| GPU（Graphics Processing Unit） | 图形处理器，本手册的主战场 |
| CPU（Central Processing Unit） | 中央处理器，负责逻辑与调度 |
| SM（Streaming Multiprocessor） | 流多处理器，GPU 的「小核」，你的卡有 24 个 |
| CUDA core | 计算核心，执行浮点运算 |
| Tensor Core | 张量核心，专做矩阵乘加的加速硬件 |
| warp | 线程束，32 个线程，GPU 真正的调度单位 |
| warp scheduler | warp 调度器，每个 SM 有 4 个 |
| SIMT（Single Instruction Multiple Threads） | 单指令多线程，warp 内 32 线程同步执行同一指令 |
| FMA（Fused Multiply-Add） | 融合乘加，一次算 `a*b+c` |

### 编程模型

| 术语 | 含义 |
| --- | --- |
| kernel | 在 GPU 上并行执行的函数，用 `__global__` 声明 |
| thread | 线程，最小执行单位 |
| block（thread block） | 线程块，可共享内存、可同步 |
| grid | 网格，一次 kernel 启动的全部线程 |
| `threadIdx` / `blockIdx` | 线程 / 线程块的编号 |
| `blockDim` / `gridDim` | 每块线程数 / 块总数 |
| occupancy | 占用率，SM 上实际驻留 warp 数占上限的比例 |
| launch | 启动 kernel，写作 `<<<grid, block>>>` |
| eligible | 就绪：warp 下一条指令的条件已满足，可被发射 |

### 内存与性能

| 术语 | 含义 |
| --- | --- |
| register | 寄存器，线程私有，最快 |
| shared memory | 共享内存，block 内共享，可手工缓存 |
| global memory | 全局显存（DRAM/HBM），最慢 |
| DRAM / HBM | 显存芯片，高带宽内存 |
| L1 / L2 cache | 一级 / 二级缓存，硬件自动管理 |
| L1TEX | L1 缓存 / 纹理单元这条访存通路 |
| bandwidth | 带宽，每秒能搬多少字节 |
| latency | 延迟，一次操作要等多久 |
| throughput | 吞吐，单位时间完成多少工作 |
| coalescing | 合并访存，warp 内连续地址访问被合并 |
| tiling | 分块，把数据搬到片上反复使用 |
| reuse | 数据复用，同一份数据被多次使用 |
| vectorization | 向量化，如用 `float4` 一次搬 16 字节 |
| fusion | 算子融合，多个算子合成一个 kernel |
| stall | 停滞，warp 等待条件满足 |
| scoreboard dependency | 记分牌依赖，后序指令要等前序结果 |
| predicated off | 谓词屏蔽，指令被条件关掉不执行 |
| IPC（Instructions Per Cycle） | 每周期指令数，衡量流水线利用率 |
| ILP（Instruction-Level Parallelism） | 指令级并行 |

### 性能模型

| 术语 | 含义 |
| --- | --- |
| FLOP | 一次浮点运算 |
| FLOP/s（FLOPS） | 每秒浮点运算次数 |
| arithmetic intensity（AI） | 算术强度，FLOP ÷ 字节数 |
| machine balance | 机器平衡点，峰值算力 ÷ 峰值带宽 |
| memory-bound | 访存受限 |
| compute-bound | 算力受限 |
| latency-bound | 延迟受限 |
| roofline | 屋顶线模型，性能上限的可视化 |
| profiler | 性能分析器，本手册用 Nsight Compute（ncu） |
| INF / WRN | profiler 的提示 / 警告级别 |
| warm-up | 热身运行，让时钟与缓存稳定 |
| thermal throttling | 热降频，温度过高时自动降频 |

## 第十一部分　参考资料

- **CUDA C++ Programming Guide**（官方）：Performance Guidelines 章节
- **CUDA C Best Practices Guide**（官方）：Occupancy 章节
- **Programming Massively Parallel Processors（PMPP）**：第 1～4 章，讲执行模型与内存层级的最佳入门书
- **Nsight Compute 文档**：Metrics / Roofline 两节，配合本手册第三、第五部分读

---

### 下一步

**reduction（归约）**——第一个需要 **共享内存（shared memory）**、**`__syncthreads` 同步** 和 **warp shuffle** 的 kernel，
它是 softmax、layernorm 的地基。

建议先完成第九部分的自测题，确认这个手册的内容真的吃透了，再进 reduction。

## 附录　完整 profiler 原始输出（存档）

命令：

```bash
ncu --set full -k vectorADD --launch-count 1 ./build/bin/vector_add
```

完整输出（含运行时的程序输出）：

```text
N = 67108864, block = 256, each array = 256.0 MiB
==PROF== Connected to process 132819 (/home/fate-whispering/projects/ai-infra/cuda_basic/build/bin/vector_add)
==PROF== Profiling "vectorADD": 0%....50%....100% - 34 passes
verify: PASS (0 mismatches)
best time           = 3.406 ms
effective bandwidth = 236.5 GB/s
==PROF== Disconnected from process 132819
[132819] vector_add@127.0.0.1
  vectorADD(const float *, const float *, float *, int) (262144, 1, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 8.9
    Section: GPU Speed Of Light Throughput
    ----------------------- ------------- ------------
    Metric Name               Metric Unit Metric Value
    ----------------------- ------------- ------------
    DRAM Frequency          cycle/nsecond         8.01
    SM Frequency            cycle/nsecond         1.55
    Elapsed Cycles                  cycle    5,268,927
    Memory Throughput                   %        90.76
    DRAM Throughput                     %        90.76
    Duration                      msecond         3.40
    L1/TEX Cache Throughput             %        17.11
    L2 Cache Throughput                 %        21.30
    SM Active Cycles                cycle 5,268,893.88
    Compute (SM) Throughput             %        13.27
    ----------------------- ------------- ------------

    INF   The kernel is utilizing greater than 80.0% of the available compute or memory performance of the device. To
          further improve performance, work will likely need to be shifted from the most utilized to another unit.
          Start by analyzing DRAM in the Memory Workload Analysis section.

    Section: Compute Workload Analysis
    -------------------- ----------- ------------
    Metric Name          Metric Unit Metric Value
    -------------------- ----------- ------------
    Executed Ipc Active   inst/cycle         0.33
    Executed Ipc Elapsed  inst/cycle         0.33
    Issue Slots Busy               %         8.29
    Issued Ipc Active     inst/cycle         0.33
    SM Busy                        %         8.29
    -------------------- ----------- ------------

    WRN   All compute pipelines are under-utilized. Either this kernel is very small or it doesn't issue enough warps
          per scheduler. Check the Launch Statistics and Scheduler Statistics sections for further details.

    Section: Memory Workload Analysis
    --------------------------- ------------ ------------
    Metric Name                  Metric Unit Metric Value
    --------------------------- ------------ ------------
    Memory Throughput           Gbyte/second       232.61
    Mem Busy                               %        15.98
    Max Bandwidth                          %        90.76
    L1/TEX Hit Rate                        %            0
    L2 Compression Success Rate            %            0
    L2 Compression Ratio                                0
    L2 Hit Rate                            %        33.38
    Mem Pipes Busy                         %        13.27
    --------------------------- ------------ ------------

    Section: Scheduler Statistics
    ---------------------------- ----------- ------------
    Metric Name                  Metric Unit Metric Value
    ---------------------------- ----------- ------------
    One or More Eligible                   %         8.28
    Issued Warp Per Scheduler                        0.08
    No Eligible                            %        91.72
    Active Warps Per Scheduler          warp         9.69
    Eligible Warps Per Scheduler        warp         0.09
    ---------------------------- ----------- ------------

    WRN   Every scheduler is capable of issuing one instruction per cycle, but for this kernel each scheduler only
          issues an instruction every 12.1 cycles. This might leave hardware resources underutilized and may lead to
          less optimal performance. Out of the maximum of 12 warps per scheduler, this kernel allocates an average of
          9.69 active warps per scheduler, but only an average of 0.09 warps were eligible per cycle.

    Section: Warp State Statistics
    ---------------------------------------- ----------- ------------
    Metric Name                              Metric Unit Metric Value
    ---------------------------------------- ----------- ------------
    Warp Cycles Per Issued Instruction             cycle       117.00
    Warp Cycles Per Executed Instruction           cycle       117.02
    Avg. Active Threads Per Warp                                   32
    Avg. Not Predicated Off Threads Per Warp                    28.80
    ---------------------------------------- ----------- ------------

    WRN   On average, each warp of this kernel spends 108.3 cycles being stalled waiting for a scoreboard dependency on
          a L1TEX (local, global, surface, texture, rtcore) operation. This represents about 92.6% of the total
          average of 117.0 cycles between issuing two instructions.

    Section: Instruction Statistics
    ---------------------------------------- ----------- ------------
    Metric Name                              Metric Unit Metric Value
    ---------------------------------------- ----------- ------------
    Avg. Executed Instructions Per Scheduler        inst   436,906.67
    Executed Instructions                           inst    41,943,040
    Avg. Issued Instructions Per Scheduler          inst   436,960.67
    Issued Instructions                             inst    41,948,224
    ---------------------------------------- ----------- ------------

    WRN   This kernel executes 0 fused and 2097152 non-fused FP32 instructions. By converting pairs of non-fused
          instructions to their fused, higher-throughput equivalent, the achieved FP32 performance could be
          increased by up to 50%.

    Section: Launch Statistics
    -------------------------------- --------------- ---------------
    Metric Name                          Metric Unit    Metric Value
    -------------------------------- --------------- ---------------
    Block Size                                                   256
    Function Cache Configuration                     CachePreferNone
    Grid Size                                                262,144
    Registers Per Thread             register/thread              16
    Shared Memory Configuration Size           Kbyte           16.38
    Driver Shared Memory Per Block       Kbyte/block            1.02
    Dynamic Shared Memory Per Block       byte/block               0
    Static Shared Memory Per Block        byte/block               0
    Threads                                   thread      67,108,864
    Waves Per SM                                            1,820.44
    -------------------------------- --------------- ---------------

    Section: Occupancy
    ------------------------------- ----------- ------------
    Metric Name                     Metric Unit Metric Value
    ------------------------------- ----------- ------------
    Block Limit SM                        block           24
    Block Limit Registers                 block           16
    Block Limit Shared Mem                block           16
    Block Limit Warps                     block            6
    Theoretical Active Warps per SM        warp           48
    Theoretical Occupancy                     %          100
    Achieved Occupancy                        %        81.04
    Achieved Active Warps Per SM           warp         38.90
    ------------------------------- ----------- ------------

    WRN   This kernel's theoretical occupancy is not impacted by any block limit. The difference between calculated
          theoretical (100.0%) and measured achieved occupancy (81.0%) can be the result of warp scheduling overheads
          or workload imbalances during the kernel execution.

    Section: Source Counters
    ------------------------- ----------- ------------
    Metric Name               Metric Unit Metric Value
    ------------------------- ----------- ------------
    Branch Instructions Ratio           %         0.15
    Branch Instructions              inst    6,291,456
    Branch Efficiency                   %          100
    Avg. Divergent Branches                          0
    ------------------------- ----------- ------------
```

> 提示：本附录只做存档，正文第三部分已经把它拆开逐段讲过了。